# Data Cleaning and Feature Engineering

This notebook:
1. Loads ratings and metadata JSONL files
2. Standardizes column names
3. Cleans metadata (price, images, title)
4. Creates user and item feature tables
5. Saves processed data as Parquet files


In [11]:
import json
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Optional, Union, List, Dict

# Set up paths (notebook runs from notebooks/ directory)
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")


Project root: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system
Data directory: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data
Processed directory: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data/processed


In [12]:
def load_jsonl(file_path: Path) -> pd.DataFrame:
    """Load JSONL file into DataFrame."""
    records = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return pd.DataFrame(records)

# Load data
ratings_file = DATA_DIR / "All_Beauty.jsonl"
metadata_file = DATA_DIR / "meta_All_Beauty.jsonl"

print("Loading ratings...")
df_ratings = load_jsonl(ratings_file)
print(f"Ratings shape: {df_ratings.shape}")
print(f"Ratings columns: {df_ratings.columns.tolist()}")

print("\nLoading metadata...")
df_meta = load_jsonl(metadata_file)
print(f"Metadata shape: {df_meta.shape}")
print(f"Metadata columns: {df_meta.columns.tolist()}")


Loading ratings...
Ratings shape: (701528, 10)
Ratings columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Loading metadata...
Metadata shape: (112590, 14)
Metadata columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']


In [13]:
# Column mapping for ratings
RATINGS_MAPPING = {
    'reviewerID': 'user_id',
    'reviewer_id': 'user_id',
    'userID': 'user_id',
    'overall': 'rating',
    'rating': 'rating',
    'asin': 'parent_asin',
    'parent_asin': 'parent_asin',
    'parentASIN': 'parent_asin',
}

# Column mapping for metadata
METADATA_MAPPING = {
    'asin': 'parent_asin',
    'parent_asin': 'parent_asin',
    'parentASIN': 'parent_asin',
    'title': 'title_clean',
    'title_clean': 'title_clean',
    'store': 'brand',
    'brand': 'brand',
    'main_category': 'main_category',
    'category': 'main_category',
    'price': 'price_float',
    'images': 'image_url',
}

# Standardize ratings columns
print("Standardizing ratings columns...")
for old_col, new_col in RATINGS_MAPPING.items():
    if old_col in df_ratings.columns and old_col != new_col:
        df_ratings[new_col] = df_ratings[old_col]
        if old_col != new_col:
            df_ratings = df_ratings.drop(columns=[old_col])

# Ensure required columns exist
required_ratings = ['user_id', 'parent_asin', 'rating']
missing_ratings = [col for col in required_ratings if col not in df_ratings.columns]
assert len(missing_ratings) == 0, f"Missing required ratings columns: {missing_ratings}"

# Ensure timestamp column for time-based split
if 'timestamp' not in df_ratings.columns and 'unixReviewTime' in df_ratings.columns:
    df_ratings['timestamp'] = df_ratings['unixReviewTime']
assert 'timestamp' in df_ratings.columns, "ratings must include 'timestamp' for time-based splits"

print(f"Ratings columns after mapping: {df_ratings.columns.tolist()}")

# Standardize metadata columns
print("\nStandardizing metadata columns...")
for old_col, new_col in METADATA_MAPPING.items():
    if old_col in df_meta.columns and old_col != new_col:
        df_meta[new_col] = df_meta[old_col]
        if old_col != new_col and old_col in df_meta.columns:
            df_meta = df_meta.drop(columns=[old_col])

# Ensure required columns exist
required_meta = ['parent_asin']
missing_meta = [col for col in required_meta if col not in df_meta.columns]
assert len(missing_meta) == 0, f"Missing required metadata columns: {missing_meta}"

print(f"Metadata columns after mapping: {df_meta.columns.tolist()}")


Standardizing ratings columns...
Ratings columns after mapping: ['rating', 'title', 'text', 'images', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Standardizing metadata columns...
Metadata columns after mapping: ['main_category', 'average_rating', 'rating_number', 'features', 'description', 'videos', 'categories', 'details', 'parent_asin', 'bought_together', 'title_clean', 'brand', 'price_float', 'image_url']


In [14]:
def parse_price(price_str: Union[str, float, None]) -> Optional[float]:
    """Parse price string to float. Handles ranges by taking average."""
    # Check for None first
    if price_str is None:
        return None
    
    # If it's numeric, check for NaN or return as float
    if isinstance(price_str, (int, float)):
        if pd.isna(price_str):
            return None
        return float(price_str)
    
    if not isinstance(price_str, str):
        return None
    
    # Remove currency symbols and whitespace
    price_str = price_str.strip().replace('$', '').replace(',', '')
    
    # Handle ranges (e.g., "$12.99 - $19.99")
    if ' - ' in price_str or '-' in price_str:
        parts = re.split(r'\s*-\s*', price_str)
        prices = []
        for part in parts:
            part = part.strip()
            match = re.search(r'(\d+\.?\d*)', part)
            if match:
                prices.append(float(match.group(1)))
        if prices:
            return sum(prices) / len(prices)
        return None
    
    # Extract first number
    match = re.search(r'(\d+\.?\d*)', price_str)
    if match:
        return float(match.group(1))
    
    return None

def extract_image_url(images: Union[List, Dict, str, None]) -> Optional[str]:
    """Extract first image URL from images field."""
    # Check for None first
    if images is None:
        return None
    
    # Check for scalar NaN (only for scalar types, not collections)
    if isinstance(images, (int, float)):
        if pd.isna(images):
            return None
    
    # If it's a string, return as is
    if isinstance(images, str):
        return images if images.strip() else None
    
    # If it's a list
    if isinstance(images, list):
        if len(images) == 0:
            return None
        
        first_item = images[0]
        
        # If list of strings
        if isinstance(first_item, str):
            return first_item
        
        # If list of dicts
        if isinstance(first_item, dict):
            # Try common keys
            for key in ['hiRes', 'hi_res', 'large', 'url', 'thumb']:
                if key in first_item and first_item[key]:
                    url = first_item[key]
                    if isinstance(url, str) and url.strip():
                        return url
            # If no common key, get first non-None value
            for val in first_item.values():
                if val and isinstance(val, str) and val.strip():
                    return val
    
    # If it's a dict
    if isinstance(images, dict):
        for key in ['hiRes', 'hi_res', 'large', 'url', 'thumb']:
            if key in images and images[key]:
                url = images[key]
                if isinstance(url, str) and url.strip():
                    return url
    
    return None

def clean_title(title: Union[str, None]) -> Optional[str]:
    """Clean title: strip spaces, remove control characters."""
    if title is None or pd.isna(title):
        return None
    
    if not isinstance(title, str):
        return str(title)
    
    # Remove control characters
    title = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', title)
    # Strip extra spaces
    title = ' '.join(title.split())
    return title.strip() if title else None

# Apply cleaning functions
print("Cleaning metadata...")

# Price
if 'price_float' not in df_meta.columns:
    df_meta['price_float'] = None
df_meta['price_float'] = df_meta.get('price', df_meta.get('price_float', None)).apply(parse_price)

# Image URL
if 'image_url' not in df_meta.columns:
    df_meta['image_url'] = None
images_col = df_meta.get('images', df_meta.get('image_url', None))
df_meta['image_url'] = images_col.apply(extract_image_url) if images_col is not None else None

# Title
if 'title_clean' not in df_meta.columns:
    title_col = df_meta.get('title', None)
    if title_col is not None:
        df_meta['title_clean'] = title_col.apply(clean_title)
    else:
        df_meta['title_clean'] = None
else:
    df_meta['title_clean'] = df_meta['title_clean'].apply(clean_title)

# Brand (use 'store' if 'brand' doesn't exist)
if 'brand' not in df_meta.columns:
    df_meta['brand'] = df_meta.get('store', None)

# Main category
if 'main_category' not in df_meta.columns:
    df_meta['main_category'] = df_meta.get('category', None)

print(f"Metadata after cleaning - shape: {df_meta.shape}")
print(f"Price non-null: {df_meta['price_float'].notna().sum()}")
print(f"Image URL non-null: {df_meta['image_url'].notna().sum()}")
print(f"Title non-null: {df_meta['title_clean'].notna().sum()}")


Cleaning metadata...
Metadata after cleaning - shape: (112590, 14)
Price non-null: 17704
Image URL non-null: 112590
Title non-null: 112578


In [15]:
# Clean ratings: ensure rating is numeric
df_ratings['rating'] = pd.to_numeric(df_ratings['rating'], errors='coerce')
required_cols = ['user_id', 'parent_asin', 'rating', 'timestamp']
df_ratings = df_ratings.dropna(subset=required_cols)

print(f"Ratings after cleaning - shape: {df_ratings.shape}")
print(f"Unique users: {df_ratings['user_id'].nunique()}")
print(f"Unique items: {df_ratings['parent_asin'].nunique()}")
print(f"Rating range: {df_ratings['rating'].min():.1f} - {df_ratings['rating'].max():.1f}")

Ratings after cleaning - shape: (701528, 9)
Unique users: 631986
Unique items: 115709
Rating range: 1.0 - 5.0


In [16]:
# Create user feature table
print("Creating user features...")
user_features = df_ratings.groupby('user_id').agg({
    'rating': ['mean', 'count']
}).reset_index()
user_features.columns = ['user_id', 'user_avg_rating', 'total_user_reviews']
user_features = user_features.sort_values('user_id').reset_index(drop=True)

print(f"User features shape: {user_features.shape}")
print(user_features.head())


Creating user features...
User features shape: (631986, 3)
                        user_id  user_avg_rating  total_user_reviews
0  AE222BBOVZIF42YOOPNBXL4UUMYA              5.0                   1
1  AE222FP7YRNFCEQ2W3ZDIGMSYTLQ              5.0                   1
2  AE222X475JC6ONXMIKZDFGQ7IAUA              5.0                   1
3  AE222Y4WTST6BUZ4J5Y2H6QMBITQ              4.0                   1
4  AE2232TEZOEWQLAFEX2NA6VBGMYQ              5.0                   1


In [17]:
# Create item feature table
print("Creating item features...")

# Average rating from interactions
item_ratings = df_ratings.groupby('parent_asin')['rating'].mean().reset_index()
item_ratings.columns = ['parent_asin', 'average_item_rating']

# Start with all items from ratings (to ensure all items have entries)
all_items_from_ratings = pd.DataFrame({'parent_asin': df_ratings['parent_asin'].unique()})

# Merge with metadata (left join to keep all items from ratings)
item_features = all_items_from_ratings.merge(
    df_meta[['parent_asin', 'brand', 'main_category', 'price_float', 'image_url', 'title_clean']],
    on='parent_asin',
    how='left'
)

# Merge with item ratings
item_features = item_features.merge(item_ratings, on='parent_asin', how='left')

# Ensure all required columns exist
required_item_cols = ['parent_asin', 'brand', 'main_category', 'average_item_rating', 'price_float', 'image_url', 'title_clean']
for col in required_item_cols:
    if col not in item_features.columns:
        item_features[col] = None

item_features = item_features[required_item_cols].copy()
item_features = item_features.sort_values('parent_asin').reset_index(drop=True)

print(f"Item features shape: {item_features.shape}")
print(item_features.head())


Creating item features...
Item features shape: (115709, 7)
  parent_asin                                              brand  \
0  0005946468                                          patanjali   
1  0123034892                                               None   
2  0124784577  WOW Organics Apple Cider Vinegar Shampoo - 300 mL   
3  0515059560                                               Jove   
4  0615675026                                                NaN   

  main_category  average_item_rating  price_float  \
0    All Beauty             5.000000          NaN   
1    All Beauty             5.000000          NaN   
2    All Beauty             4.333333          NaN   
3    All Beauty             4.000000          NaN   
4           NaN             2.000000          NaN   

                                           image_url  \
0  https://m.media-amazon.com/images/I/41wW6QVdVm...   
1  https://m.media-amazon.com/images/I/51mITa4VDJ...   
2  https://m.media-amazon.com/images/I/711hrX

In [18]:
# Save processed data
print("Saving processed data...")

# Ratings (only required columns)
ratings_cols = ['user_id', 'parent_asin', 'rating']
if 'timestamp' in df_ratings.columns:
    ratings_cols.append('timestamp')
ratings_clean = df_ratings[ratings_cols].copy()
ratings_clean_path = PROCESSED_DIR / "ratings_clean.parquet"
ratings_clean.to_parquet(ratings_clean_path, index=False)
print(f"Saved: {ratings_clean_path} (shape: {ratings_clean.shape})")

# User features
user_features_path = PROCESSED_DIR / "user_features.parquet"
user_features.to_parquet(user_features_path, index=False)
print(f"Saved: {user_features_path} (shape: {user_features.shape})")

# Item features
item_features_path = PROCESSED_DIR / "item_features.parquet"
item_features.to_parquet(item_features_path, index=False)
print(f"Saved: {item_features_path} (shape: {item_features.shape})")

print("\nData cleaning complete!")


Saving processed data...
Saved: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data/processed/ratings_clean.parquet (shape: (701528, 4))
Saved: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data/processed/user_features.parquet (shape: (631986, 3))
Saved: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data/processed/item_features.parquet (shape: (115709, 7))

Data cleaning complete!
